In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle
import os

df = pd.read_csv('../data/property_listings.csv')
print(f"Raw shape: {df.shape}")

Raw shape: (105, 8)


In [2]:
import re

# Price
df['price_clean'] = (
    df['Price']
    .str.replace('KSh', '', regex=False)
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(' ', '', regex=False)
    .str.strip()
)
df['price_clean'] = pd.to_numeric(df['price_clean'], errors='coerce')

# Floor size
df['floor_size_clean'] = (
    df['Floor Size']
    .str.replace('m²', '', regex=False)
    .str.replace('m2', '', regex=False)
    .str.strip()
)
df['floor_size_clean'] = pd.to_numeric(df['floor_size_clean'], errors='coerce')

# Dedup
df = df.drop_duplicates(subset=['Title', 'Price', 'Address', 'Floor Size'])

# Property type
df['property_type'] = df['Title'].str.extract(
    r'(Apartment|House|Villa|Townhouse|Studio|Penthouse)', flags=re.IGNORECASE
)
df['property_type'] = df['property_type'].fillna('Other').str.title()

# Price per sqm
df['price_per_sqm'] = df['price_clean'] / df['floor_size_clean']

# Bedrooms + bathrooms
df['bedrooms'] = pd.to_numeric(df['Bedrooms'], errors='coerce')
df['bathrooms'] = df['Bathrooms']

print(f"Clean shape: {df.shape}")

Clean shape: (66, 14)


In [3]:
# Label encode Location
le_location = LabelEncoder()
df['location_encoded'] = le_location.fit_transform(df['Location'])

# Label encode property type
le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['property_type'])

# Show mapping so we know what each number means
print("Location encoding:")
for i, label in enumerate(le_location.classes_):
    print(f"  {i} → {label}")

print("\nProperty type encoding:")
for i, label in enumerate(le_type.classes_):
    print(f"  {i} → {label}")

Location encoding:
  0 → Kiambu Road
  1 → Kileleshwa
  2 → Kilimani
  3 → Lower Kabete
  4 → Riverside
  5 → Westlands

Property type encoding:
  0 → Apartment
  1 → House
  2 → Townhouse


In [4]:
# Select final features for training
feature_cols = ['bedrooms', 'bathrooms', 'floor_size_clean', 'price_per_sqm', 
                'location_encoded', 'type_encoded']
target_col = 'price_clean'

train_df = df[feature_cols + [target_col]].dropna()
print(f"Training dataset shape: {train_df.shape}")
print(train_df.head())

# Save
os.makedirs('../data', exist_ok=True)
train_df.to_csv('../data/train_ready.csv', index=False)

# Save encoders for later use (inference)
os.makedirs('../models', exist_ok=True)
with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump({'location': le_location, 'property_type': le_type}, f)

print("\n✅ train_ready.csv saved")
print("✅ encoders.pkl saved")

Training dataset shape: (66, 7)
   bedrooms  bathrooms  floor_size_clean  price_per_sqm  location_encoded  \
0         4        4.0               211  127962.085308                 5   
1         4        5.5               514  247276.264591                 5   
3         2        2.0               140   91785.714286                 5   
4         2        2.0                88  102272.727273                 2   
6         3        4.5               230  171739.130435                 5   

   type_encoded  price_clean  
0             0     27000000  
1             0    127100000  
3             0     12850000  
4             0      9000000  
6             0     39500000  

✅ train_ready.csv saved
✅ encoders.pkl saved
